In [ ]:
import matplotlib.pyplot as plt
from types import SimpleNamespace
from gould_2026.datasets import LDS, NestDynamicsUFunction, get_nest_u_to_s_given_stim
import numpy as np
from learn_s_hat_toy import n_rotations, stims_per_rotation, stim_magnitude, noise_variance, transitions_per_rotation
from gould_2026.plotting import Palette, paper_plot_context
from functools import partial
from pathlib import Path
from typing import Literal

In [ ]:
output_1_wire_trace = None
output_2_surface = None
output_3_heatmap = None
plot_type = NestDynamicsUFunction.baffled

In [ ]:
plot_type = NestDynamicsUFunction(plot_type)

output_1_wire_trace = Path(output_1_wire_trace) if output_1_wire_trace is not None else None
output_2_surface = Path(output_2_surface) if output_2_surface is not None else None
output_3_heatmap = Path(output_3_heatmap) if output_3_heatmap is not None else None


In [ ]:
%matplotlib inline
rng = np.random.default_rng(5)

stim_steps = {16, 32, 80, 150}

show_toy_n_turns = 10

_, Y, stim = LDS.run_nest_dynamical_system(rotations=show_toy_n_turns, stims_per_rotation=.75, stim_magnitude=stim_magnitude, rng=rng, u_function=plot_type, noise=noise_variance, radius=15)

with paper_plot_context():
    fig1, ax = plt.subplots(subplot_kw=dict(projection="3d"), figsize=(1,1))
    ax.plot(Y[:, 0], Y[:, 1], Y[:, 2], color='k')


    ax.axis('equal')
    if plot_type != 'curvy_flips':
        ax.view_init(elev=24, azim=147, roll=0)
    else:
        ax.view_init(elev=-24, azim=147, roll=0)
    ax.axis((-18., 15., -15., 15., -10., 10.))
    ax.xaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
    ax.yaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
    ax.zaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
    ax.xaxis._axinfo["grid"]['color'] = (1, 1, 1, 0)
    ax.yaxis._axinfo["grid"]['color'] = (1, 1, 1, 0)
    ax.zaxis._axinfo["grid"]['color'] = (1, 1, 1, 0)

    # for i in range(3):
    #     start = np.array([-10,-15,7.5])
    #     delta = np.zeros(3)
    #     delta[i] = 5
    #     end = start + delta
    #     ax.plot([start[0], end[0]], [start[1], end[1]], [start[2], end[2]], c=f'C{i}')

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])
    # ax.set_axis_off()

    if output_1_wire_trace is not None:
        fig1.savefig(output_1_wire_trace)


In [ ]:
fig2, ax = plt.subplots()

ax.plot(Y[:,:2], color='k', alpha=.5)
ax.plot(Y[:,2], color='k')

for step in stim_steps:
    ax.axvline(step, color='red', alpha=.5, linestyle='--')



In [ ]:

lds = SimpleNamespace(B=SimpleNamespace(shape=(3,)))
f = get_nest_u_to_s_given_stim(u_function=plot_type, stim_magnitude=stim_magnitude, transition_time=1, transitions_per_rotation=transitions_per_rotation)
f = partial(f, lds=lds, i=1.0*transitions_per_rotation, rng=rng)

s = np.where((stim == 1).any(axis=1))[0]
evaluated = []
for y in Y[s-1]:
    evaluated.append(f(state=y)[2])


depth = 500

f_evaluated = np.zeros((depth, depth))
XX = np.linspace(-20, 20, depth)
YY = np.linspace(-20, 20, depth)
XX, YY = np.meshgrid(XX, YY)

for i in range(depth):
    for j in range(depth):
        state = np.array([XX[i,j], YY[i,j], 0])
        f_evaluated[i,j] = f(state=state)[2]




In [ ]:
%matplotlib inline
azim = 11 * 10


In [ ]:

with paper_plot_context():
    fig = plt.figure(figsize=(2,2))
    ax = fig.add_subplot(1, 1, 1, projection='3d')
    ax.plot(Y[:, 0], Y[:, 1], Y[:, 2], color='k')
    ax.view_init(elev=25, azim=azim, roll=0)
    ax.set_zlim(-10,10)
    ax.scatter(Y[s, 0], Y[s, 1], Y[s, 2], c=evaluated, cmap='plasma', vmin=-10, vmax=10)
    ax.xaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
    ax.yaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
    ax.zaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
    ax.grid(False)
    ax.set_xlim(-20,20)
    ax.set_ylim(-20,20)
    ax.tick_params(axis='x', pad=-6)
    ax.tick_params(axis='z', pad=-2)
    ax.set_yticks([])

    if output_1_wire_trace is not None:
        fig.savefig(output_1_wire_trace)

In [ ]:

with paper_plot_context():
    fig = plt.figure(figsize=(2,2))
    ax = fig.add_subplot(1, 1, 1, projection='3d')
    ax.plot_surface(XX, YY, f_evaluated, cmap='plasma', vmin=-10, vmax=10, rasterized=True)
    ax.view_init(elev=25, azim=azim, roll=0)
    ax.grid(False)
    ax.xaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
    ax.yaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
    ax.zaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
    ax.tick_params(axis='x', pad=-6)
    ax.tick_params(axis='z', pad=-2)
    ax.set_yticks([])
    if output_2_surface is not None:
        fig.savefig(output_2_surface)


In [ ]:

with paper_plot_context():
    fig = plt.figure(figsize=(2,2))
    ax = fig.add_subplot(1, 1, 1)
    ax.plot(Y[:,0], Y[:,1], 'k', alpha=0)
    ax.pcolormesh(XX, YY, f_evaluated, cmap='plasma', shading='auto', vmin=-10, vmax=10, rasterized=True)
    ax.plot(Y[:, 0], Y[:, 1], color='k')
    ax.scatter(Y[s, 0], Y[s, 1], c=evaluated, cmap='plasma', vmin=-10, vmax=10, zorder=10, edgecolors='gray')
    ax.set_xlim(-20,20)
    ax.set_ylim(-20,20)
    ax.axis('scaled')

    if output_3_heatmap is not None:
        fig.savefig(output_3_heatmap)


